# Vectorial Focusing of a Structured Beam

Cleaned and documented version of the numerical simulation developed for my BSc thesis at the Photonics Laboratory, ETH Zürich.

The notebook constructs two orthogonal first-order Hermite–Gaussian input modes, transforms them into radial and azimuthal polarization components, and evaluates the vectorial high-NA focusing integral to obtain the focal electric-field components \(E_x\), \(E_y\), and \(E_z\).

The numerical model and physical conventions are preserved from the original thesis code; the implementation has been reorganized for readability and reproducibility.

## 1. Imports and simulation parameters

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from numpy.polynomial.hermite import hermval

# Optical parameters
NA = 1.4
N_MEDIUM = 1.518
N_INPUT = 1.0
WAVELENGTH = 1.0

# Pupil discretization
NUM_PUPIL_PIXELS = 201

# Structured input beam
FILLING_FACTOR = 1.5
BEAM_WAIST = FILLING_FACTOR

# Focal-plane sampling
FOCUS_MIN = -0.5
FOCUS_MAX = 0.5
FOCUS_STEP = 0.01
FOCUS_Z = 0.0

# Output directory
DATA_DIR = Path("results")
DATA_DIR.mkdir(exist_ok=True)

## 2. Helper functions

The focal field is evaluated using the vectorial full-wave focusing integral. The pupil field is first transformed into Cartesian components and then numerically integrated over the angular coordinates \(\theta\) and \(\phi\).

In [ ]:
def integrate_focal_field(
    e_inf_x, e_inf_y, e_inf_z, k, theta, phi,
    x_focus, y_focus, z, dtheta, dphi
):
    """Numerically integrate the three Cartesian focal-field components."""
    e_x = np.zeros_like(x_focus, dtype=complex)
    e_y = np.zeros_like(x_focus, dtype=complex)
    e_z = np.zeros_like(x_focus, dtype=complex)

    rho = np.sqrt(x_focus**2 + y_focus**2)
    phi_focus = np.arctan2(y_focus, x_focus)

    sin_theta = np.sin(theta)
    cos_theta = np.cos(theta)

    for i in range(x_focus.shape[0]):
        for j in range(x_focus.shape[1]):
            phase = np.exp(
                1j * k * (
                    z * cos_theta
                    + rho[i, j] * sin_theta
                    * np.cos(phi - phi_focus[i, j])
                )
            )
            integration_weight = phase * sin_theta * dphi * dtheta

            e_x[i, j] = np.sum(e_inf_x * integration_weight)
            e_y[i, j] = np.sum(e_inf_y * integration_weight)
            e_z[i, j] = np.sum(e_inf_z * integration_weight)

    return e_x, e_y, e_z


def focus_vectorial_field(
    e_azimuthal,
    e_radial,
    *,
    n_input=N_INPUT,
    n_medium=N_MEDIUM,
    wavelength=WAVELENGTH,
    numerical_aperture=NA,
    focus_min=FOCUS_MIN,
    focus_max=FOCUS_MAX,
    focus_step=FOCUS_STEP,
    z=FOCUS_Z,
):
    """Transform the pupil field and evaluate the high-NA focusing integral."""
    num_pixels = e_azimuthal.shape[0]

    theta_max = np.arcsin(numerical_aperture / n_medium)
    theta_values = np.linspace(0.0, np.pi / 2.0, num_pixels)
    phi_values = np.linspace(-np.pi, np.pi, num_pixels)
    theta, phi = np.meshgrid(theta_values, phi_values)

    pupil_mask = theta < theta_max
    dtheta = theta_max / num_pixels
    dphi = 2.0 * np.pi / num_pixels
    k = 2.0 * np.pi * n_medium / wavelength

    # Original thesis model assumes unit transmission coefficients.
    t_s = np.ones_like(e_azimuthal)
    t_p = np.ones_like(e_radial)

    apodization = np.sqrt((n_input / n_medium) * np.cos(theta)) * pupil_mask

    e_inf_x = (
        t_s * e_azimuthal * (-np.sin(phi))
        + t_p * e_radial * np.cos(phi) * np.cos(theta)
    ) * apodization

    e_inf_y = (
        t_s * e_azimuthal * np.cos(phi)
        + t_p * e_radial * np.sin(phi) * np.cos(theta)
    ) * apodization

    e_inf_z = (-t_p * e_radial * np.sin(theta)) * apodization

    # Preserve the spatial scaling used in the original implementation.
    scaled_min = focus_min * n_medium**2
    scaled_max = focus_max * n_medium**2
    scaled_step = focus_step * n_medium**2

    x_focus, y_focus = np.mgrid[
        scaled_min:scaled_max:scaled_step,
        scaled_min:scaled_max:scaled_step,
    ]

    e_x, e_y, e_z = integrate_focal_field(
        e_inf_x, e_inf_y, e_inf_z, k, theta, phi,
        x_focus, y_focus, z, dtheta, dphi
    )

    return e_x, e_y, e_z, x_focus, y_focus


def plot_field_components(e_x, e_y, e_z, x_focus, y_focus):
    """Plot normalized Cartesian focal-field intensities."""
    i_x = np.abs(e_x) ** 2
    i_y = np.abs(e_y) ** 2
    i_z = np.abs(e_z) ** 2
    i_total = i_x + i_y + i_z

    extent = [x_focus.min(), x_focus.max(), y_focus.min(), y_focus.max()]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for ax, intensity, title in zip(
        axes, (i_x, i_y, i_z), (r"$|E_x|^2$", r"$|E_y|^2$", r"$|E_z|^2$")
    ):
        image = ax.imshow(
            intensity / i_total.max(),
            origin="lower",
            extent=extent,
            aspect="equal",
        )
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        fig.colorbar(image, ax=ax)

    fig.tight_layout()
    plt.show()


def plot_total_intensity(
    e_x, e_y, e_z, x_focus, y_focus, longitudinal_weight=1.0
):
    """Plot total intensity and central x/y cross sections."""
    intensity = (
        np.abs(e_x) ** 2
        + np.abs(e_y) ** 2
        + longitudinal_weight * np.abs(e_z) ** 2
    )

    extent = [x_focus.min(), x_focus.max(), y_focus.min(), y_focus.max()]
    axis = np.linspace(x_focus.min(), x_focus.max(), intensity.shape[0])
    center = intensity.shape[0] // 2

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    image = axes[0].imshow(
        intensity, origin="lower", extent=extent, aspect="equal"
    )
    axes[0].set_title(
        "Total intensity"
        if longitudinal_weight == 1.0
        else rf"$|E_x|^2+|E_y|^2+{longitudinal_weight}|E_z|^2$"
    )
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")
    fig.colorbar(image, ax=axes[0])

    normalized = intensity / intensity.max()
    axes[1].plot(axis, normalized[:, center], label="x cross section")
    axes[1].plot(axis, normalized[center, :], "--", label="y cross section")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_xlim(axis.min(), axis.max())
    axes[1].set_xlabel("Distance")
    axes[1].set_ylabel("Normalized intensity")
    axes[1].set_title("Central cross sections")
    axes[1].legend()

    fig.tight_layout()
    plt.show()

## 3. Pupil coordinate system

In [ ]:
theta_values = np.linspace(0.0, np.pi / 2.0, NUM_PUPIL_PIXELS)
phi_values = np.linspace(-np.pi, np.pi, NUM_PUPIL_PIXELS)
theta, phi = np.meshgrid(theta_values, phi_values)

theta_max = np.arcsin(NA / N_MEDIUM)
pupil_mask = theta < theta_max
pupil_radius = np.sin(theta)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, quantity, title in zip(
    axes,
    (pupil_radius * pupil_mask, phi * pupil_mask, theta * pupil_mask),
    (r"$\sin\theta$", r"$\phi$", r"$\theta$"),
):
    image = ax.imshow(quantity, origin="lower", aspect="auto")
    ax.set_title(title)
    ax.set_xlabel(r"$\theta$ index")
    ax.set_ylabel(r"$\phi$ index")
    fig.colorbar(image, ax=ax)

fig.tight_layout()
plt.show()

## 4. Structured input beam

Two orthogonal first-order Hermite–Gaussian modes are constructed. Their superposition is subsequently expressed in the radial/azimuthal polarization basis before focusing.

In [ ]:
grid = np.linspace(-1.0, 1.0, 100)
x_input, y_input = np.meshgrid(grid, grid)

focal_length = BEAM_WAIST / (FILLING_FACTOR * np.sin(theta_max))
radius_input = np.sqrt(x_input**2 + y_input**2)

hg_x_polynomial = hermval(x_input, [0, 1])
hg_y_polynomial = hermval(y_input, [0, 1])

gaussian_envelope = np.exp(
    -(radius_input * focal_length) ** 2 / BEAM_WAIST**2
)

hg_x = hg_x_polynomial * gaussian_envelope
hg_y = hg_y_polynomial * gaussian_envelope

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
input_intensities = (
    np.abs(hg_x) ** 2,
    np.abs(hg_y) ** 2,
    np.abs(hg_x) ** 2 + np.abs(hg_y) ** 2,
)
titles = (r"$|HG_x|^2$", r"$|HG_y|^2$", r"$|HG_x|^2 + |HG_y|^2$")

for ax, intensity, title in zip(axes, input_intensities, titles):
    image = ax.imshow(
        intensity, origin="lower", extent=[-1, 1, -1, 1], aspect="equal"
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(image, ax=ax)

fig.tight_layout()
plt.show()

## 5. Radial and azimuthal pupil-field components

In [ ]:
pupil_radius = np.sin(theta)
x_pupil = pupil_radius * np.cos(phi)
y_pupil = pupil_radius * np.sin(phi)

hg_x_pupil = hermval(x_pupil, [0, 1]) * np.exp(
    -(pupil_radius * focal_length) ** 2 / BEAM_WAIST**2
)
hg_y_pupil = hermval(y_pupil, [0, 1]) * np.exp(
    -(pupil_radius * focal_length) ** 2 / BEAM_WAIST**2
)

e_azimuthal = -hg_x_pupil * np.sin(phi) + hg_y_pupil * np.cos(phi)
e_radial = hg_x_pupil * np.cos(phi) + hg_y_pupil * np.sin(phi)

## 6. Vectorial high-NA focusing

The following cell evaluates the full vectorial focusing integral. With the original \(201\times201\) pupil grid and fine focal-plane sampling, this calculation can take some time because the integral is evaluated independently at every focal-plane point.

In [ ]:
e_focus_x, e_focus_y, e_focus_z, x_focus, y_focus = focus_vectorial_field(
    e_azimuthal,
    e_radial,
    numerical_aperture=NA,
    focus_min=FOCUS_MIN,
    focus_max=FOCUS_MAX,
    focus_step=FOCUS_STEP,
)

print("Focal-field grid:", e_focus_x.shape)

## 7. Save numerical results

In [ ]:
np.savetxt(DATA_DIR / f"E_focus_X_NA={NA:.1f}.txt", e_focus_x)
np.savetxt(DATA_DIR / f"E_focus_Y_NA={NA:.1f}.txt", e_focus_y)
np.savetxt(DATA_DIR / f"E_focus_Z_NA={NA:.1f}.txt", e_focus_z)

print(f"Saved focal fields to: {DATA_DIR.resolve()}")

## 8. Focal-field results

In [ ]:
plot_field_components(e_focus_x, e_focus_y, e_focus_z, x_focus, y_focus)

plot_total_intensity(
    e_focus_x, e_focus_y, e_focus_z, x_focus, y_focus
)

# Alternative longitudinal-field weighting used in the original analysis.
plot_total_intensity(
    e_focus_x,
    e_focus_y,
    e_focus_z,
    x_focus,
    y_focus,
    longitudinal_weight=0.2,
)